# Turing Game — Análisis de Partidas

Notebook para extraer, estructurar y analizar los ~31 juegos target.

**Dos modos de carga:**
1. **Directo de Supabase** (preferido) — usa `supabase-py` con la service role key
2. **Desde CSVs** — exporta las queries 3, 4 y 5 del archivo SQL como CSV y cárgalos aquí

---

In [ ]:
# pip install supabase pandas matplotlib seaborn
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict

sns.set_theme(style="darkgrid")
pd.set_option('display.max_colwidth', 120)

## 0. Configuración — Lista de Game IDs Target

In [ ]:
TARGET_GAME_IDS = [
    # Reemplaza con tus 31 game_ids reales
    "GAME_ID1", "GAME_ID2", "GAME_ID3", "GAME_ID4", "GAME_ID5",
    "GAME_ID6", "GAME_ID7", "GAME_ID8", "GAME_ID9", "GAME_ID10",
    "GAME_ID11", "GAME_ID12", "GAME_ID13", "GAME_ID14", "GAME_ID15",
    "GAME_ID16", "GAME_ID17", "GAME_ID18", "GAME_ID19", "GAME_ID20",
    "GAME_ID21", "GAME_ID22", "GAME_ID23", "GAME_ID24", "GAME_ID25",
    "GAME_ID26", "GAME_ID27", "GAME_ID28", "GAME_ID29", "GAME_ID30",
    "GAME_ID31",
]

print(f"Target games: {len(TARGET_GAME_IDS)}")

## 1. Carga de Datos

### Opción A: Directo desde Supabase

In [ ]:
from supabase import create_client

SUPABASE_URL = ""       # tu URL de Supabase
SUPABASE_KEY = ""       # service_role key (no anon) para bypass de RLS

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

games_raw = sb.table("games").select("*").in_("id", TARGET_GAME_IDS).execute().data
messages_raw = sb.table("messages").select("*").in_("game_id", TARGET_GAME_IDS).order("created_at").execute().data
lessons_raw = sb.table("lessons").select("*").order("created_at").execute().data

df_games = pd.DataFrame(games_raw)
df_messages = pd.DataFrame(messages_raw)
df_lessons = pd.DataFrame(lessons_raw)

print(f"Games: {len(df_games)}, Messages: {len(df_messages)}, Lessons: {len(df_lessons)}")

### Opción B: Desde CSVs exportados

Exporta las queries 3, 4 y 5 del archivo `game_analysis.sql` como CSV desde el SQL Editor de Supabase y colócalos en `analysis/data/`.

In [ ]:
# Descomenta si usas CSVs:
# df_games = pd.read_csv("data/games.csv")
# df_messages = pd.read_csv("data/messages.csv")
# df_lessons = pd.read_csv("data/lessons.csv")
# print(f"Games: {len(df_games)}, Messages: {len(df_messages)}, Lessons: {len(df_lessons)}")

## 2. Limpieza y Preparación

In [ ]:
for col in ["created_at", "started_at", "ended_at"]:
    df_games[col] = pd.to_datetime(df_games[col], utc=True)

df_messages["created_at"] = pd.to_datetime(df_messages["created_at"], utc=True)
df_lessons["created_at"] = pd.to_datetime(df_lessons["created_at"], utc=True)
df_lessons["updated_at"] = pd.to_datetime(df_lessons["updated_at"], utc=True)

df_games["duracion_min"] = (
    (df_games["ended_at"] - df_games["started_at"]).dt.total_seconds() / 60
)

df_games = df_games.sort_values("created_at").reset_index(drop=True)
df_games["game_number"] = range(1, len(df_games) + 1)

df_games[["game_number", "id", "status", "claude_slot", "guess_correct", "duracion_min"]].head(10)

## 3. Tasa de Engaño (Deception Rate)

In [ ]:
ended = df_games[df_games["status"] == "ended"].copy()

ended["resultado"] = ended["guess_correct"].map({
    True: "P1 acierta (detectó a Claude)",
    False: "P1 falla (Claude engañó)",
})
ended["resultado"] = ended["resultado"].fillna("Timeout (cuenta como fallo de P1)")

resumen = ended["resultado"].value_counts()
total = len(ended)
enganos = len(ended[ended["guess_correct"] != True])

print(f"Total juegos terminados: {total}")
print(f"Tasa de engaño (Claude engañó a P1): {enganos}/{total} = {enganos/total*100:.1f}%")
print()
print(resumen.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#e74c3c" if "falla" in r or "Timeout" in r else "#2ecc71" for r in resumen.index]
resumen.plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Cantidad de juegos")
ax.set_title(f"Resultado de los {total} juegos — Tasa de engaño: {enganos/total*100:.0f}%")
plt.tight_layout()
plt.show()

## 4. Extracción Estructurada — JSON por Juego

Construye el objeto principal de análisis: un diccionario con toda la información de cada juego organizada.

In [ ]:
def build_game_data(game_row, messages_df, lessons_df):
    """Construye el objeto estructurado de un juego individual."""
    gid = game_row["id"]
    claude_slot = game_row["claude_slot"]
    human_slot = "right" if claude_slot == "left" else "left"

    game_msgs = messages_df[messages_df["game_id"] == gid].copy()
    game_msgs = game_msgs.sort_values("created_at")

    in_game = game_msgs[game_msgs["slot"].notna()]
    claude_chat = in_game[in_game["slot"] == claude_slot]
    human_chat = in_game[in_game["slot"] == human_slot]
    feedback_msgs = game_msgs[(game_msgs["slot"].isna()) & (game_msgs["sender"] == "p1")]

    def format_chat(chat_df):
        return [
            {
                "sender": row["sender"],
                "content": row["content"],
                "timestamp": row["created_at"].isoformat() if pd.notna(row["created_at"]) else None,
            }
            for _, row in chat_df.iterrows()
        ]

    resultado = "no_terminado"
    if game_row["status"] == "ended":
        if game_row["guess_correct"] is True or game_row["guess_correct"] == True:
            resultado = "interrogador_acierta"
        elif game_row["guess_correct"] is False or game_row["guess_correct"] == False:
            resultado = "interrogador_falla"
        else:
            resultado = "timeout"

    game_lessons = lessons_df[lessons_df["game_id"] == gid]

    persona = game_row.get("claude_persona")
    if isinstance(persona, str):
        try:
            persona = json.loads(persona)
        except (json.JSONDecodeError, TypeError):
            pass

    return {
        "game_id": gid,
        "game_number": game_row["game_number"],
        "hora_inicio": game_row["started_at"].isoformat() if pd.notna(game_row["started_at"]) else None,
        "hora_fin": game_row["ended_at"].isoformat() if pd.notna(game_row["ended_at"]) else None,
        "duracion_minutos": round(game_row["duracion_min"], 2) if pd.notna(game_row["duracion_min"]) else None,
        "resultado": resultado,
        "claude_slot": claude_slot,
        "p1_guess_left": game_row["p1_guess_left"],
        "p1_guess_right": game_row["p1_guess_right"],
        "claude_persona": persona,
        "chat_p1_claude": format_chat(claude_chat),
        "chat_p1_human": format_chat(human_chat),
        "feedback_p1": [r["content"] for _, r in feedback_msgs.iterrows()],
        "total_mensajes_claude_chat": len(claude_chat),
        "total_mensajes_human_chat": len(human_chat),
        "lessons_generadas": [
            {
                "content": r["content"],
                "weight": r["weight"],
                "created_at": r["created_at"].isoformat() if pd.notna(r["created_at"]) else None,
            }
            for _, r in game_lessons.iterrows()
        ],
    }


all_games_data = {}
for _, game_row in df_games.iterrows():
    gdata = build_game_data(game_row, df_messages, df_lessons)
    all_games_data[gdata["game_id"]] = gdata

print(f"Juegos extraídos: {len(all_games_data)}")
print(f"Ejemplo de keys por juego: {list(list(all_games_data.values())[0].keys())}")

In [ ]:
# Guardar a disco para reutilizar sin re-consultar Supabase
with open("data/all_games_structured.json", "w", encoding="utf-8") as f:
    json.dump(all_games_data, f, ensure_ascii=False, indent=2, default=str)

print("Guardado en data/all_games_structured.json")

## 5. Exploración de un Juego Individual

Selecciona un juego y revisa sus chats completos.

In [ ]:
def print_game_summary(game_id: str):
    g = all_games_data[game_id]
    print(f"={'='*60}")
    print(f"JUEGO: {g['game_id']} (#{g['game_number']})")
    print(f"Resultado: {g['resultado']}")
    print(f"Duración: {g['duracion_minutos']} min")
    print(f"Claude slot: {g['claude_slot']}")
    persona = g['claude_persona']
    if isinstance(persona, dict):
        print(f"Persona: {persona.get('name', '?')} — {persona.get('major', '?')}")
    print(f"Guess: left={g['p1_guess_left']}, right={g['p1_guess_right']}")
    print(f"Lessons generadas: {len(g['lessons_generadas'])}")
    if g['feedback_p1']:
        print(f"Feedback P1: {g['feedback_p1'][0][:100]}...")
    print()

    print(f"--- Chat P1 vs Claude ({len(g['chat_p1_claude'])} msgs) ---")
    for msg in g["chat_p1_claude"]:
        label = msg["sender"].upper()
        print(f"  [{label}] {msg['content'][:150]}")
    print()

    print(f"--- Chat P1 vs Human ({len(g['chat_p1_human'])} msgs) ---")
    for msg in g["chat_p1_human"]:
        label = msg["sender"].upper()
        print(f"  [{label}] {msg['content'][:150]}")


# Imprime el primer juego como ejemplo
first_id = list(all_games_data.keys())[0]
print_game_summary(first_id)

## 6. Análisis de Tiempos de Respuesta — Claude vs P2

In [ ]:
def compute_response_times(game_data):
    """Calcula tiempos de respuesta para ambos chats."""
    results = []
    for chat_key, label in [("chat_p1_claude", "claude"), ("chat_p1_human", "human")]:
        msgs = game_data[chat_key]
        for i in range(1, len(msgs)):
            prev = msgs[i - 1]
            curr = msgs[i]
            if prev["timestamp"] and curr["timestamp"]:
                t_prev = pd.Timestamp(prev["timestamp"])
                t_curr = pd.Timestamp(curr["timestamp"])
                delta_sec = (t_curr - t_prev).total_seconds()
                if 0 < delta_sec < 600:  # filtrar outliers
                    results.append({
                        "game_id": game_data["game_id"],
                        "chat": label,
                        "responder": curr["sender"],
                        "response_time_sec": delta_sec,
                        "msg_length": len(curr["content"]),
                    })
    return results

all_response_times = []
for gdata in all_games_data.values():
    all_response_times.extend(compute_response_times(gdata))

df_rt = pd.DataFrame(all_response_times)

if not df_rt.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, responder_group, title in [
        (axes[0], df_rt[df_rt["responder"].isin(["claude", "p2"])], "Tiempo de respuesta: Claude vs P2"),
        (axes[1], df_rt, "Distribución general"),
    ]:
        if not responder_group.empty:
            sns.boxplot(data=responder_group, x="responder", y="response_time_sec", ax=ax)
            ax.set_title(title)
            ax.set_ylabel("Segundos")

    plt.tight_layout()
    plt.show()

    print("\nEstadísticas por responder:")
    print(df_rt.groupby("responder")["response_time_sec"].describe().round(1))

## 7. Análisis de Longitud de Mensajes

In [ ]:
all_msg_stats = []
for gdata in all_games_data.values():
    for chat_key, label in [("chat_p1_claude", "claude_chat"), ("chat_p1_human", "human_chat")]:
        for msg in gdata[chat_key]:
            all_msg_stats.append({
                "game_id": gdata["game_id"],
                "resultado": gdata["resultado"],
                "chat": label,
                "sender": msg["sender"],
                "length": len(msg["content"]),
                "words": len(msg["content"].split()),
            })

df_ml = pd.DataFrame(all_msg_stats)

if not df_ml.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    witnesses = df_ml[df_ml["sender"].isin(["claude", "p2"])]
    sns.histplot(data=witnesses, x="words", hue="sender", bins=30, ax=axes[0], alpha=0.6)
    axes[0].set_title("Distribución de palabras por mensaje — Claude vs P2")
    axes[0].set_xlabel("Palabras por mensaje")

    sns.boxplot(data=witnesses, x="sender", y="words", hue="resultado", ax=axes[1])
    axes[1].set_title("Palabras por mensaje según resultado")

    plt.tight_layout()
    plt.show()

    print("\nPromedio de palabras por sender:")
    print(witnesses.groupby("sender")["words"].describe().round(1))

## 8. Evolución de Lessons — Línea Temporal

In [ ]:
from matplotlib.lines import Line2D
import numpy as np

# --- Construir timeline: lessons, victorias de Claude, peso acumulado ---

WEIGHT_MIN = 7  # rango real observado de importancia

lessons_timeline = []
cumulative_wins = 0
cumulative_lessons = 0
cumulative_weighted = 0.0

lessons_sorted = df_lessons.sort_values("created_at")

for _, game_row in df_games.iterrows():
    game_time = game_row["created_at"]

    new_lessons = lessons_sorted[
        (lessons_sorted["created_at"] <= game_time)
    ]
    cumulative_lessons = len(new_lessons)
    cumulative_weighted = new_lessons["weight"].sum() - (WEIGHT_MIN - 1) * len(new_lessons)

    is_claude_win = (
        game_row["status"] == "ended"
        and (game_row["guess_correct"] is False or game_row["guess_correct"] == False)
    )
    if is_claude_win:
        cumulative_wins += 1

    lessons_timeline.append({
        "game_number": game_row["game_number"],
        "game_id": game_row["id"],
        "guess_correct": game_row["guess_correct"],
        "lessons_count": cumulative_lessons,
        "claude_wins": cumulative_wins,
        "weighted_lessons": cumulative_weighted,
        "created_at": game_time,
    })

df_timeline = pd.DataFrame(lessons_timeline)

# --- Gráfica con 3 líneas y doble eje Y ---

fig, ax1 = plt.subplots(figsize=(14, 6))

color_lessons = "#3498db"
color_wins = "#2ecc71"
color_weighted = "#e67e22"

ax1.plot(df_timeline["game_number"], df_timeline["lessons_count"],
         "-o", color=color_lessons, label="Lessons acumuladas (conteo)", markersize=5, linewidth=2)
ax1.plot(df_timeline["game_number"], df_timeline["claude_wins"],
         "-s", color=color_wins, label="Victorias de Claude (acum.)", markersize=5, linewidth=2)
ax1.set_xlabel("Juego #", fontsize=12)
ax1.set_ylabel("Conteo acumulado", fontsize=12, color="black")
ax1.tick_params(axis="y")

ax2 = ax1.twinx()
ax2.plot(df_timeline["game_number"], df_timeline["weighted_lessons"],
         "--^", color=color_weighted, label="Lessons ponderadas (Σ peso − 6)",
         markersize=5, linewidth=2, alpha=0.85)
ax2.set_ylabel("Importancia acumulada (peso − 6 por lesson)", fontsize=11, color=color_weighted)
ax2.tick_params(axis="y", labelcolor=color_weighted)

for _, row in df_timeline.iterrows():
    if pd.notna(row["guess_correct"]):
        bg = "#2ecc71" if not row["guess_correct"] else "#e74c3c"
        ax1.axvspan(row["game_number"] - 0.4, row["game_number"] + 0.4,
                     color=bg, alpha=0.12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
extra = [
    Line2D([0], [0], color="#2ecc71", linewidth=6, alpha=0.2, label="Claude engañó"),
    Line2D([0], [0], color="#e74c3c", linewidth=6, alpha=0.2, label="P1 detectó a Claude"),
]
ax1.legend(handles=lines1 + lines2 + extra, loc="upper left", fontsize=9)

ax1.set_title("Correlación: Lessons acumuladas vs Victorias de Claude", fontsize=14)
ax1.set_xticks(df_timeline["game_number"])
fig.tight_layout()
plt.show()

# --- Correlación numérica ---
valid = df_timeline.dropna(subset=["guess_correct"])
if len(valid) > 2:
    corr_count = valid["lessons_count"].corr(valid["claude_wins"])
    corr_weighted = valid["weighted_lessons"].corr(valid["claude_wins"])
    print(f"Correlación (Pearson) lessons_count vs claude_wins:    {corr_count:.3f}")
    print(f"Correlación (Pearson) weighted_lessons vs claude_wins: {corr_weighted:.3f}")

In [ ]:
# Tabla detallada: qué lesson se creó después de qué juego
print("Lessons ordenadas por creación:")
print("=" * 80)
for _, lesson in df_lessons.sort_values("created_at").iterrows():
    source_game = lesson["game_id"]
    game_info = ""
    if source_game and source_game in all_games_data:
        g = all_games_data[source_game]
        game_info = f" (Juego #{g['game_number']}, resultado: {g['resultado']})"
    print(f"\n[weight={lesson['weight']}] {lesson['content']}")
    print(f"  Creada: {lesson['created_at']}  |  Game: {source_game}{game_info}")

## 9. Análisis Adicionales

### 9.1 Duración del juego vs Resultado

In [ ]:
ended_with_dur = ended.dropna(subset=["duracion_min"]).copy()

if not ended_with_dur.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.stripplot(data=ended_with_dur, x="resultado", y="duracion_min", ax=ax, size=8, jitter=True)
    ax.set_title("Duración del juego según resultado")
    ax.set_ylabel("Minutos")
    ax.set_xlabel("")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

    print("\nDuración promedio por resultado:")
    print(ended_with_dur.groupby("resultado")["duracion_min"].describe().round(2))

### 9.2 Profundidad de Conversación (nº de intercambios)

In [ ]:
depth_data = []
for gdata in all_games_data.values():
    depth_data.append({
        "game_id": gdata["game_id"],
        "game_number": gdata["game_number"],
        "resultado": gdata["resultado"],
        "msgs_claude_chat": gdata["total_mensajes_claude_chat"],
        "msgs_human_chat": gdata["total_mensajes_human_chat"],
        "total_msgs": gdata["total_mensajes_claude_chat"] + gdata["total_mensajes_human_chat"],
        "ratio_claude_human": (
            gdata["total_mensajes_claude_chat"] / max(gdata["total_mensajes_human_chat"], 1)
        ),
    })

df_depth = pd.DataFrame(depth_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_melted = df_depth.melt(
    id_vars=["game_number", "resultado"],
    value_vars=["msgs_claude_chat", "msgs_human_chat"],
    var_name="chat", value_name="mensajes"
)
sns.barplot(data=df_melted, x="game_number", y="mensajes", hue="chat", ax=axes[0])
axes[0].set_title("Mensajes por chat por juego")
axes[0].tick_params(axis="x", rotation=90)

sns.scatterplot(data=df_depth, x="total_msgs", y="resultado", hue="resultado", s=100, ax=axes[1])
axes[1].set_title("Total de mensajes vs Resultado")

plt.tight_layout()
plt.show()

### 9.3 Análisis de Persona — ¿Qué personas fueron más convincentes?

In [ ]:
persona_data = []
for gdata in all_games_data.values():
    p = gdata.get("claude_persona")
    if isinstance(p, dict):
        persona_data.append({
            "game_id": gdata["game_id"],
            "nombre": p.get("name", "?"),
            "carrera": p.get("major", "?"),
            "semestre": p.get("semester", "?"),
            "resultado": gdata["resultado"],
            "engano": gdata["resultado"] == "interrogador_falla",
        })

df_persona = pd.DataFrame(persona_data)
if not df_persona.empty:
    print("Personas usadas por Claude:")
    print(df_persona[["game_id", "nombre", "carrera", "semestre", "resultado"]].to_string(index=False))
    print()
    if "carrera" in df_persona.columns:
        print("Tasa de engaño por carrera:")
        print(df_persona.groupby("carrera")["engano"].mean().sort_values(ascending=False).round(2))

### 9.4 Análisis Lingüístico Básico — Patrones de detección

In [ ]:
import re

def linguistic_features(text: str) -> dict:
    """Extrae features lingüísticos simples de un mensaje."""
    return {
        "chars": len(text),
        "words": len(text.split()),
        "exclamations": text.count("!"),
        "questions": text.count("?"),
        "emojis": len(re.findall(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]', text)),
        "uppercase_ratio": sum(1 for c in text if c.isupper()) / max(len(text), 1),
        "has_typo_indicators": bool(re.search(r'\b(jaja|xd|nmms|wey|neta|osea|ogt|chido|alv)\b', text, re.I)),
        "starts_lowercase": text[0].islower() if text else False,
        "ends_without_punct": not text.rstrip().endswith(('.', '!', '?', ')')) if text.strip() else False,
    }

ling_data = []
for gdata in all_games_data.values():
    for chat_key, who in [("chat_p1_claude", "claude"), ("chat_p1_human", "human")]:
        for msg in gdata[chat_key]:
            if msg["sender"] in ("claude", "p2"):
                features = linguistic_features(msg["content"])
                features["witness"] = who
                features["sender"] = msg["sender"]
                features["game_id"] = gdata["game_id"]
                features["resultado"] = gdata["resultado"]
                ling_data.append(features)

df_ling = pd.DataFrame(ling_data)

if not df_ling.empty:
    print("Comparación lingüística Claude vs P2 (promedios):")
    print("=" * 60)
    comparison = df_ling.groupby("witness")[
        ["words", "exclamations", "questions", "emojis",
         "uppercase_ratio", "starts_lowercase", "ends_without_punct"]
    ].mean().round(3).T
    print(comparison)
    print()
    print("Uso de slang mexicano:")
    print(df_ling.groupby("witness")["has_typo_indicators"].mean().round(3))

### 9.5 Tasa de Engaño Acumulada (rolling window)

In [ ]:
df_timeline["engano"] = df_timeline["guess_correct"].apply(
    lambda x: 1 if x is False or x == False else (0 if x is True or x == True else None)
)
df_timeline_valid = df_timeline.dropna(subset=["engano"]).copy()

if len(df_timeline_valid) > 3:
    df_timeline_valid["tasa_acumulada"] = (
        df_timeline_valid["engano"].expanding().mean() * 100
    )
    window = min(5, len(df_timeline_valid))
    df_timeline_valid["tasa_rolling"] = (
        df_timeline_valid["engano"].rolling(window=window, min_periods=2).mean() * 100
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df_timeline_valid["game_number"], df_timeline_valid["tasa_acumulada"],
            "b-o", label="Tasa acumulada", markersize=5)
    ax.plot(df_timeline_valid["game_number"], df_timeline_valid["tasa_rolling"],
            "r--s", label=f"Rolling (window={window})", markersize=4, alpha=0.7)
    ax.axhline(y=50, color="gray", linestyle=":", alpha=0.5, label="50% (azar)")
    ax.set_xlabel("Juego #")
    ax.set_ylabel("Tasa de engaño (%)")
    ax.set_title("Evolución de la tasa de engaño de Claude")
    ax.legend()
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()

### 9.6 Primer Mensaje — ¿La primera impresión importa?

In [ ]:
first_msg_data = []
for gdata in all_games_data.values():
    for chat_key, who in [("chat_p1_claude", "claude"), ("chat_p1_human", "human")]:
        witness_msgs = [m for m in gdata[chat_key] if m["sender"] in ("claude", "p2")]
        if witness_msgs:
            first = witness_msgs[0]
            first_msg_data.append({
                "game_id": gdata["game_id"],
                "witness": who,
                "first_msg": first["content"],
                "first_msg_words": len(first["content"].split()),
                "resultado": gdata["resultado"],
            })

df_first = pd.DataFrame(first_msg_data)
if not df_first.empty:
    print("Primeros mensajes de Claude vs P2:")
    print("=" * 60)
    for _, row in df_first.iterrows():
        emoji = "🤖" if row["witness"] == "claude" else "👤"
        result_emoji = "✅" if row["resultado"] == "interrogador_falla" else "❌"
        print(f"{emoji} [{row['game_id']}] {result_emoji} \"{row['first_msg'][:100]}\"")

### 9.7 Tabla Resumen Final

In [ ]:
summary_rows = []
for gdata in all_games_data.values():
    persona = gdata.get("claude_persona", {}) or {}
    summary_rows.append({
        "#": gdata["game_number"],
        "ID": gdata["game_id"],
        "Resultado": gdata["resultado"],
        "Duración (min)": gdata["duracion_minutos"],
        "Msgs Claude": gdata["total_mensajes_claude_chat"],
        "Msgs Human": gdata["total_mensajes_human_chat"],
        "Persona": persona.get("name", "?") if isinstance(persona, dict) else "?",
        "Lessons creadas": len(gdata["lessons_generadas"]),
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

print("\n" + "=" * 60)
total = len(df_summary)
fallas = len(df_summary[df_summary["Resultado"] == "interrogador_falla"])
print(f"TASA DE ENGAÑO FINAL: {fallas}/{total} = {fallas/total*100:.1f}%")